#  `#pragma omp for`
* `#pragma omp parallel` merely creates a team of threads executing the same statement
* In this sense, `#pragma omp parallel` alone cannot make a program run faster with multiple cores
* A program can be made faster only when you _divide_ the work among threads (work-sharing)
* `#pragma omp for` lets you divide iterations of a loop into threads created by `#pragma omp parallel`

# <font color="green"> Problem 1 :  How does `#pragma omp for` divide iterations to threads?</font>
* Execute the following cell and observe which iteration is executed by which thread
* Based on the observation, change the number of iterations and threads and predict the mapping between iterations and threads


In [ ]:
%%writefile omp_for.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
#pragma omp parallel
  {
    printf("I am thread %d in a team of %d threads\n",
           omp_get_thread_num(), omp_get_num_threads());
#pragma omp for
    for (int i = 0; i < 24; i++) {
      usleep(100 * 1000 * i);
      printf("iteration %d executed by thread %d\n", i, omp_get_thread_num());
      fflush(stdout);
    }
  }
  return 0;
}

In [ ]:
%%bash
clang -fopenmp omp_for.c -o omp_for
# nvc -mp omp_for.c -o omp_for

In [ ]:
%%bash
OMP_NUM_THREADS=4 ./omp_for

## 1-1. for loops allowed by `#pragma omp for`
* There is a severe syntax restriction on the kind of for loops `#pragma omp for` can apply for
* See [Canonical Loop Form](https://www.openmp.org/spec-html/5.2/openmpsu40.html#x63-1260002.9.1) for the spec
* In short, it should look like `for (var = _init_; var < _limit_; var += _inc_)` where _init_, _limit_, and _inc_ are all loop-invariant (do not change throughout the loop)

## 1-2. Combined pragma (parallel + for)
* `#pragma omp parallel` and `#pragma omp for` are often used together
* If `#pragma omp for` immediately follows `#pragma omp parallel`, they can be combined into a single pragma `#pragma omp parallel for`


In [ ]:
%%writefile omp_parallel_for.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  double t0 = omp_get_wtime();
#pragma omp parallel for
  for (int i = 0; i < 24; i++) {
    usleep(100 * 1000 * i);     /* sleep 100 x i milliseconds */
    printf("iteration %d executed by thread %d\n", i, omp_get_thread_num());
    fflush(stdout);
  }
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}

In [ ]:
%%bash
clang -fopenmp omp_parallel_for.c -o omp_parallel_for
# nvc -mp omp_parallel_for.c -o omp_parallel_for

In [ ]:
%%bash
OMP_NUM_THREADS=4 ./omp_parallel_for

# 2. Scheduling a work-sharing for loop
* As you witnessed, the default scheduling policy in our environment (may be implementation dependent) seems static scheduling (assign roughly the same number of contiguous iterations to each thread)
* Is it enough? Clearly, it does not do a good job when iterations take a different amount of time
* You can change the policy by [schedule clause](https://www.openmp.org/spec-html/5.2/openmpsu41.html#x64-1290002.9.2)

## 2-1. Visualizing scheduling
* The program below executes the function `iter_fun`
```
#pragma omp parallel for
  for (long i = 0; i < L; i++) {
    iter_fun(a, b, i, M, N, R, T);
  }
```

* `iter_fun(a, b, i, M, N, R, T)` repeats x = a x + b many (M * N) times and record time every N iterations


In [ ]:
%%writefile omp_sched_rec.c
#include <err.h>
#include <sched.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <unistd.h>
#include <omp.h>

long cur_time_ns() {
  struct timespec ts[1];
  if (clock_gettime(CLOCK_REALTIME, ts) == -1) err(1, "clock_gettime");
  return ts->tv_sec * 1000000000L + ts->tv_nsec;
}

typedef struct {
  double x;
  int thread[2];
  int cpu[2];
} record_t;

/* the function for an iteration
   perform
   x = a x + b
   (M * N) times and record current time
   every N iterations to T.
   record thread and cpu to R.
 */
void iter_fun(double a, double b, long i, long M, long N,
              record_t * R, long * T) {
  // initial value (not important)
  double x = i;
  // record in T[i * M] ... T[(i+1) * M - 1]
  T = &T[i * M];
  // record starting thread/cpu
  R[i].thread[0] = omp_get_thread_num();
  R[i].cpu[0] = sched_getcpu();
  // repeat a x + b many times.
  // record time every N iterations
  for (long j = 0; j < M; j++) {
    T[j] = cur_time_ns();
    for (long k = 0; k < N; k++) {
      x = a * x + b;
    }
  }
  // record ending SM (must be = thread0)
  R[i].thread[1] = omp_get_thread_num();
  R[i].cpu[1] = sched_getcpu();
  // record result, just so that the computation is not
  // eliminated by the compiler
  R[i].x = x;
}

void dump(record_t * R, long * T, long L, long M, long t0) {
  long k = 0;
  for (long i = 0; i < L; i++) {
    printf("i=%ld x=%f thread0=%d cpu0=%d thread1=%d cpu1=%d",
           i, R[i].x, R[i].thread[0], R[i].cpu[0], R[i].thread[1], R[i].cpu[1]);
    for (long j = 0; j < M; j++) {
      printf(" %ld", T[k] - t0);
      k++;
    }
    printf("\n");
  }
}

int main(int argc, char ** argv) {
  int idx = 1;
  long L   = (idx < argc ? atol(argv[idx]) : 100);  idx++;
  long M   = (idx < argc ? atol(argv[idx]) : 100);  idx++;
  long N   = (idx < argc ? atol(argv[idx]) : 100);  idx++;
  double a = (idx < argc ? atof(argv[idx]) : 0.99); idx++;
  double b = (idx < argc ? atof(argv[idx]) : 1.00); idx++;
  record_t * R = (record_t *)calloc(L, sizeof(record_t));
  long * T = (long *)calloc(L * M, sizeof(long));
  long t0 = cur_time_ns();
#pragma omp parallel for
  for (long i = 0; i < L; i++) {
    iter_fun(a, b, i, M, N, R, T);
  }
  long t1 = cur_time_ns();
  printf("%ld nsec\n", t1 - t0);
  dump(R, T, L, M, t0);
  return 0;
}

In [ ]:
%%bash
clang -fopenmp -D_GNU_SOURCE omp_sched_rec.c -o omp_sched_rec
# nvc -mp omp_sched_rec.c -o omp_sched_rec

In [ ]:
%%bash
OMP_NUM_THREADS=4 ./omp_sched_rec > a.dat

* Execute the following cell to visialize it
* In the graph,
  * horizontal axis is the time from the start in nanosecond
  * vertical axis is the iteration number
  * the color represents the thread that executed the iteration


In [ ]:
import sched_vis
sched_vis.sched_plt(["a.dat"])
# sched_vis.sched_plt(["a.dat"], start_t=1.5e7, end_t=2.0e7)

# <font color="green"> Problem 2 :  Understanding scheduling by visualization</font>
* Add `schedule` clause to the program (`schedule(runtime)` allows you to set the schedule in the command line)
* Change the number of threads and schedule and observe how iterations are executed
* Set the number of threads very large (higher than the physical number of cores) and see what happens
  * Hint : you can get the number of cores by `nproc` command 
* In the above program, each iteration performs exactly the same amount of computation (i.e., x = a x + b (M * N) times), thus takes almost exactly the same time
* See what happens if this is not the case
  * Specifically, make iteration `i` repeats x = a x + b (M * (i * N)) times (i.e., change the inner loop in `iter_fun` to `for (long k = 0; k < i * N; k++) { ...`)

* `sched_plt` function below takes optional parameters `start_t` and `end_t` specifying the horizontal range to display
* If you zoom _very_ closely to a particular point, you can see individual points and intervals between them, from which you can deduce how long it takes to perform `x = a x + b` once


In [ ]:
%%bash
nproc

# <font color="green"> Problem 3 :  Specifying the scheduling policy by schedule clause</font>
* In the following (artificial) loop, iteration _i_ roughly sleeps for (100 x _i_) milliseconds and this is almost exactly the time it takes
1. predict the executing time of the parallel for loop with the default (static) scheduling policy
1. vary the scheduling policy and reason about their execution times


In [ ]:
%%writefile omp_schedule.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  double t0 = omp_get_wtime();
  /* ----- add schedule clause below ----- */
#pragma omp parallel for
  for (int i = 0; i < 12; i++) {
    usleep(100 * 1000 * i);     /* sleep 100 x i milliseconds */
    printf("iteration %d executed by thread %d\n", i, omp_get_thread_num());
    fflush(stdout);
  }
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}

In [ ]:
%%bash
clang -fopenmp omp_schedule.c -o omp_schedule
# nvc -mp omp_schedule.c -o omp_schedule

In [ ]:
%%bash
OMP_NUM_THREADS=4 ./omp_schedule

* Predict the execution time with the default (static) policy
* Predict the execution time with the dynamic policy?
* Compare them with what you observed

* Explain your reasoning below


# 3. Collapse clause
* `#pragma omp for` can specify a [collapse clause](https://www.openmp.org/spec-html/5.2/openmpsu41.html#x64-1290002.9.2) to apply work-sharing for a limited type of nested loops
* With a `#pragma omp for clause(2)`, OpenMP considers the doubly-nested loop that comes after this pragma the subject of work-sharing (i.e., distribute iterations of the doubly-nested loop to threads); you must have a _perfectly-nested_, rectangular doubly-nested loop after this clause
* A perfectly-nested loop is a nested loop whose outer loops (all loops except for the innermost one) do not have any statement except the inner loop. e.g.
```
for (i = 0; i < 100; i++) {
  for (j = 0; j < 100; j++) {
    S(i,j);
  }
}
```
is perfectly nested whereas
```
for (i = 0; i < 100; i++) {
  S;
  for (j = 0; j < 100; j++) {
    T;
  }
}
```
is not.  
* A perfectly nested loop is conceptually a flat loop with a mechanical transformation.
```
for (ij = 0; ij < 100 * 100; ij++) {
  i = ij / 100;
  j = ij % 100;
  S(i,j);
}
```
* A rectangular loop is a loop whose iteration counts of inner loops never depend on outer loops.  For example,
```
for (i = 0; i < 100; i++) {
  for (j = 0; j < i; j++) {
    S(i,j);
  }
}
```
is not a rectangular loop.
* Generally speaking, OpenMP `#pragma omp parallel` + `#pragma omp for` cannot handle nested parallelism very well, but collapse clause alleviates the problem to some extent
* Consider using tasks below for more general form of nested parallelism

# <font color="green"> Problem 4 :  Apply collapse and schedule</font>
* Apply collapse and schedule to the following loop
* Trick: write `schedule(runtime)` and you can change the scheduling policy at execution time by setting environment variable `OMP_SCHEDULE=` in the command line. See [OMP_SCHEDULE environment variable](https://www.openmp.org/spec-html/5.2/openmpsu41.html#x64-1370002.9.2.1) for details
* Reason about the execution time of various schedule policies and with/without collapse


In [ ]:
%%writefile omp_collapse.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  double t0 = omp_get_wtime();
  /* apply collapse and schedule */
#pragma omp parallel for
  for (int i = 0; i < 5; i++) {
    for (int j = 0; j < 5; j++) {
      usleep(100 * 1000 * (i + j));
      printf("iteration (%d, %d) executed by thread %d\n", i, j, omp_get_thread_num());
      fflush(stdout);
    }
  }
  double t1 = omp_get_wtime();
  printf("%f sec\n", t1 - t0);
  return 0;
}




In [ ]:
%%bash
clang -fopenmp omp_collapse.c -o omp_collapse
# nvc -mp omp_collapse.c -o omp_collapse

In [ ]:
%%bash
OMP_NUM_THREADS=3 ./omp_collapse